In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from IPython.display import SVG
import os
#   Not normally recommended
#   Done this way only because I do not normally use the subfolders as a module system
#   These folders are meant to store data as an intermediate processing step
sgdloader = __import__('hdb-properties.sg_open_data_loader').sg_open_data_loader

In [2]:
def load_acra(filename:str):
    df = sgdloader.getDataInFrame(
        filename=filename,
        foldname='acra-information',
        low_memory=False,
        na_values=('na'),
        usecols=[
            'uen',
            'entity_name',
            'entity_type_description',
            'entity_status_description',
            'registration_incorporation_date',
            'uen_issue_date',
            'address_type',
            'block',
            'street_name',
            'level_no',
            'unit_no',
            'building_name',
            'postal_code',
            'other_address_line1',
            'other_address_line2',
            'primary_ssic_code',
            'primary_ssic_description',
            'secondary_ssic_code',
            'secondary_ssic_description'
        ],
        parse_dates=['registration_incorporation_date','uen_issue_date'])
    return df

In [3]:
acra = load_acra(filename='acra-information-on-corporate-entities-others.csv')
acra = pd.concat([acra,*[load_acra(filename='acra-information-on-corporate-entities-{}.csv'.format(c)) for c in sorted(list('qwertyuiopasdfghjklzxcvbnm'))]],axis=0)
acra.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1849108 entries, 0 to 14977
Data columns (total 19 columns):
 #   Column                           Dtype         
---  ------                           -----         
 0   uen                              object        
 1   entity_name                      object        
 2   entity_type_description          object        
 3   entity_status_description        object        
 4   registration_incorporation_date  datetime64[ns]
 5   uen_issue_date                   datetime64[ns]
 6   address_type                     object        
 7   block                            object        
 8   street_name                      object        
 9   level_no                         object        
 10  unit_no                          object        
 11  building_name                    object        
 12  postal_code                      object        
 13  other_address_line1              float64       
 14  other_address_line2              flo

In [4]:
acra.loc[acra['uen'].fillna('').str.contains(r'(?i:pf)',regex=True)]

,uen,entity_name,entity_type_description,entity_status_description,registration_incorporation_date,uen_issue_date,address_type,block,street_name,level_no,unit_no,building_name,postal_code,other_address_line1,other_address_line2,primary_ssic_code,primary_ssic_description,secondary_ssic_code,secondary_ssic_description
1286,T04PF0001D,A B NG & CO.,PAF,Removed (Applied To Be Revoked),2004-04-01 00:00:00,2008-12-16,LOCAL,5001,BEACH ROAD,04,15,GOLDEN MILE COMPLEX,199588,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
1530,T04PF0766F,A CHEONG & CO,PAF,Removed (Applied To Be Revoked),2004-04-01 00:00:00,2008-12-16,LOCAL,260A,LORONG CHUAN,NaN,NaN,BOUNDARY GARDENS,556757,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
2246,T10PF0017A,A K KOH & ASSOCIATES,PAF,Live,2010-12-07 00:00:00,2010-12-07,LOCAL,336,SMITH STREET,05,311,NEW BRIDGE CENTRE,050336,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
6592,S85PF0003K,A. GUAN & CO.,PAF,Live,1985-11-05 00:00:00,2008-12-16,LOCAL,4,LENG KEE ROAD,04,01,SIS BUILDING,159088,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
6616,S82PF0004B,A. L. LEE & CO.,PAF,Live,1982-11-10 00:00:00,2008-12-16,LOCAL,151,CHIN SWEE ROAD,13,01,MANHATTAN HOUSE,169876,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2432,T16PF0009G,ZAPIENT,PAF,Removed (Applied To Be Revoked),2016-10-03 09:32:11,2016-10-03,LOCAL,809,FRENCH ROAD,06,152,KITCHENER COMPLEX,200809.0,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
3074,T21PF0002B,ZE AUDIT & ASSURANCE,PAF,Removed (Applied To Be Revoked),2021-02-27 16:25:55,2021-02-27,LOCAL,14,ROBINSON ROAD,08,01A,FAR EAST FINANCE BUILDING,48545.0,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
7649,T10PF0021H,ZHEN PUBLIC ACCOUNTING FIRM,PAF,Live,2010-12-31 00:00:00,2010-12-31,LOCAL,171,CHIN SWEE ROAD,04,08,CES CENTRE,169877.0,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN
9555,T10PF0009K,ZHONGLEI SINGAPORE,PAF,Live,2010-06-18 00:00:00,2010-06-18,LOCAL,12,NEW INDUSTRIAL ROAD,03,03B,MORNINGSTAR CENTRE,536202.0,NaN,NaN,69201,ACCOUNTING AND AUDITING SERVICES (EXCLUDING ON...,NaN,NaN


In [13]:
#   Address List from OneMap (via street names)
om1 = sgdloader.getDataInFrame(filename='onemap_addresses_street_name.json', foldname='hdb-properties', todrop=['searchval','address','longtitude'], dropduplicatesby=['blk_no','road_name','building','postal'])
om1.head(5)

,blk_no,road_name,building,postal,x,y,latitude,longitude
0,,SENGKANG WEST WAY,NIL,NIL,32921.874596,41923.353788,1.395414,103.877545
1,448B,SENGKANG WEST WAY,FERNVALE CREST,792448,32451.617581,41702.837206,1.393420,103.873320
2,449,SENGKANG WEST WAY,FERNVALE CREST,790449,32428.281632,41733.874329,1.393701,103.873110
3,450,SENGKANG WEST WAY,FERNVALE CREST,790450,32462.516265,41789.160134,1.394200,103.873418
4,450A,SENGKANG WEST WAY,FERNVALE CREST,791450,32472.408115,41818.756773,1.394468,103.873507


In [14]:
#   Address List from OneMap
om2 = sgdloader.getDataInFrame(filename='onemap_addresses_postal.json', foldname='hdb-properties', todrop=['searchval','address','longtitude'], dropduplicatesby=['blk_no','road_name','building','postal'])
om2.head(5)

,blk_no,road_name,building,postal,x,y,latitude,longitude
0,101A,BAYFRONT AVENUE,TEMPORARY SITE OFFICE,018895,30485.511362,28685.612665,1.275697,103.855652
1,1,STRAITS BOULEVARD,SINGAPORE CHINESE CULTURAL CENTRE,018906,29809.365407,28700.236127,1.275829,103.849576
2,11A,STRAITS BOULEVARD,TEMPORARY SITE OFFICE,018907,30041.838898,28602.987244,1.274950,103.851665
3,2,CENTRAL BOULEVARD,IOI CENTRAL BOULEVARD TOWERS,018916,30024.919852,29136.807026,1.279777,103.851513
4,21,PARK STREET,MARINA BAY MRT STATION,018925,30369.006866,28753.531902,1.276311,103.854605


In [15]:
om = pd.concat((om1,om2),axis=0).drop_duplicates(subset=['blk_no','road_name','postal'], keep='first')
del om1, om2
om

,blk_no,road_name,building,postal,x,y,latitude,longitude
0,,SENGKANG WEST WAY,NIL,NIL,32921.874596,41923.353788,1.395414,103.877545
1,448B,SENGKANG WEST WAY,FERNVALE CREST,792448,32451.617581,41702.837206,1.393420,103.873320
2,449,SENGKANG WEST WAY,FERNVALE CREST,790449,32428.281632,41733.874329,1.393701,103.873110
3,450,SENGKANG WEST WAY,FERNVALE CREST,790450,32462.516265,41789.160134,1.394200,103.873418
4,450A,SENGKANG WEST WAY,FERNVALE CREST,791450,32472.408115,41818.756773,1.394468,103.873507
...,...,...,...,...,...,...,...,...
143143,6,BEGONIA CRESCENT,HOCK SWEE HILL,809975,31275.765316,41219.415645,1.389048,103.862754
143144,7,BEGONIA CRESCENT,HOCK SWEE HILL,809976,31224.054766,41212.949466,1.388990,103.862289
143145,8,BEGONIA CRESCENT,HOCK SWEE HILL,809977,31294.702207,41212.952644,1.388990,103.862924
143146,9,BEGONIA CRESCENT,HOCK SWEE HILL,809978,31235.158364,41229.149980,1.389136,103.862389


In [16]:
acra.shape

(1849108, 19)

In [17]:
acra.merge(right=om.dropna(subset=['postal'],how='any'), left_on=['postal_code'], right_on=['postal'], how='left')

,uen,entity_name,entity_type_description,entity_status_description,registration_incorporation_date,uen_issue_date,address_type,block,street_name,level_no,...,secondary_ssic_code,secondary_ssic_description,blk_no,road_name,building,postal,x,y,latitude,longitude
0,53278504E,!3MED,Business,Cancelled,2014-10-13 04:08:56,2014-10-21,LOCAL,296,PUNGGOL CENTRAL,05,...,86904.0,HOME HEALTHCARE SERVICES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53311591D,!DEABUNDANCE,Business,Cancelled,2015-07-23 21:03:54,2015-07-24,LOCAL,50,LORONG 28 GEYLANG,03,...,46900.0,WHOLESALE TRADE OF A VARIETY OF GOODS WITHOUT ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201542182E,!DEAS CREATIVE STUDIO PTE. LTD.,Local Company,Live Company,2015-12-02 22:34:13,2015-12-03,LOCAL,8,BURN ROAD,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,52892494W,!WEB A-GO-GO!,Business,NaN,1999-05-12 00:00:00,2008-09-11,LOCAL,5,KISMIS ROAD,NaN,...,62090.0,OTHER INFORMATION TECHNOLOGY AND COMPUTER SERV...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,53030238A,""" L "" PINE GARDEN SCAPE & GEN CONTRACTOR",Business,NaN,2004-09-30 20:38:13,2008-09-12,LOCAL,10,ANSON ROAD,27,...,52212.0,MOTOR VEHICLE TOWING SERVICES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1953903,53101445B,ZZZ TRADING,Business,Live,2007-09-25 00:07:04,2008-09-13,LOCAL,445,TAMPINES STREET 42,02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1953904,201822294H,ZZZATELIER SINGAPORE PTE. LTD.,Local Company,Live Company,2018-07-02 11:22:09,2018-07-02,LOCAL,293,OCEAN DRIVE,03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1953905,53019964E,ZZZUNIOR PUB,Business,NaN,2004-05-05 17:29:39,2008-09-12,LOCAL,4,DUXTON HILL,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1953906,202219255H,ZZZZIP PTE. LTD.,Local Company,Live Company,2022-06-02 00:00:00,2022-06-02,LOCAL,16,RAFFLES QUAY,41,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
